# Extract SAE Features - Gemma Scope 2

This notebook demonstrates how to use Gemma Scope 2 SAEs for feature extraction:
1. Load Gemma 3 model with memory validation
2. Load Gemma Scope 2 SAE (directly from HuggingFace)
3. Extract sparse feature activations
4. Compare features between Markdown vs Plain Text prompts

SAEs decompose dense activations into ~60 sparse, interpretable features
per token (L0 ≈ 60), providing memory-efficient mechanistic interpretability.

In [1]:
import os

import torch
from dotenv import load_dotenv
from huggingface_hub import login

from model_evaluation.main_agent import (
    GemmaModelConfig,
    MemoryTracker,
    compare_feature_activations,
    extract_sae_features,
    load_gemma_model,
    load_gemma_scope_sae,
    print_memory_usage,
    print_model_info,
    visualize_token_activations,
    visualize_top_features_per_token,
)
from model_evaluation.main_agent.example_prompts import (
    get_markdown_prompts,
    get_plain_prompts,
)

load_dotenv()
torch.set_grad_enabled(False)

if os.getenv("HF_TOKEN"):
    login(token=os.getenv("HF_TOKEN"))

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
MODEL_ID = "google/gemma-3-12b-it"

print_model_info(MODEL_ID, quantization="int4")


════════════════════════════════════════════════════════════
📊 Gemma 3 12B Model Info
════════════════════════════════════════════════════════════

🏗️  Architecture:
    Parameters:     12.0B
    Layers:         48 (8 global, 40 local)
    Heads:          16 (KV: 8)
    Sliding Window: 1024

💾 Memory (int4, 32k context):
    Weights:        8.0 GB
    KV Cache:       2.3 GB
    Attn (global):  64.0 GB/layer
════════════════════════════════════════════════════════════



In [3]:
config = GemmaModelConfig(
    model_id=MODEL_ID,
    quantization="int4",
    max_context_length=8192,
)

print(f"\n{'=' * 60}")
print(f"Loading {config.model_id}")
print(f"Quantization: {config.quantization or 'None (bf16)'}")
print(f"{'=' * 60}")

with MemoryTracker(f"Loading {config.model_id}"):
    model, tokenizer = load_gemma_model(config)
    model.eval()

device = str(next(model.parameters()).device)
print(f"Model loaded on: {device}")


Loading google/gemma-3-12b-it
Quantization: int4
📦 Using Unsloth pre-quantized model: unsloth/gemma-3-12b-it-bnb-4bit


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]


──────────────────────────────────────────────────
📊 Memory Tracking: Loading google/gemma-3-12b-it
──────────────────────────────────────────────────
  ⏱️  Duration:    5.81s
  💾 RSS Change:  +272.98 MB
  🖥️  CUDA Change: +7437.72 MB
  📈 Peak CUDA:   11622.00 MB
──────────────────────────────────────────────────

Model loaded on: cuda:0


In [4]:
# Load SAE for residual stream at a late layer (layer 41 for 12B model)
# This captures abstract concepts the model has developed
with MemoryTracker("Loading Gemma Scope 2 SAE"):
    sae, sae_config = load_gemma_scope_sae(
        model_size="12b",
        model_type="it",  # instruction-tuned (matches our model)
        layer=41,  # ~85% depth for abstract concepts (available: 12, 24, 31, 41)
        width="16k",  # 16k features (memory efficient)
        l0_size="medium",  # ~60 active features per token
        device=device,
    )

print("\nSAE Info:")
print(f"  Features: {sae_config.d_sae}")
print(f"  Layer: {sae_config.layer}")
print(f"  Width: {sae_config.width}")

📦 Loading SAE from: google/gemma-scope-2-12b-it
   File: resid_post/layer_41_width_16k_l0_medium/params.safetensors
✅ SAE loaded: 16384 features, layer 41

──────────────────────────────────────────────────
📊 Memory Tracking: Loading Gemma Scope 2 SAE
──────────────────────────────────────────────────
  ⏱️  Duration:    0.36s
  💾 RSS Change:  +477.43 MB
  🖥️  CUDA Change: +480.14 MB
  📈 Peak CUDA:   8398.00 MB
──────────────────────────────────────────────────


SAE Info:
  Features: 16384
  Layer: 41
  Width: 16k


In [5]:
test_prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": "What is the capital of France?"}],
    tokenize=False,
    add_generation_prompt=True,
)

with MemoryTracker("SAE Feature Extraction"):
    result = extract_sae_features(
        model=model,
        tokenizer=tokenizer,
        sae=sae,
        sae_config=sae_config,
        text=test_prompt,
        max_new_tokens=50,
        top_k=10,
    )

print("\n📊 SAE Feature Extraction Results:")
num_answer_tokens = len(result.tokens) - result.prompt_len
print(f"  Tokens: {len(result.tokens)} (Prompt: {result.prompt_len}, Answer: {num_answer_tokens})")
print(f"  L0 (avg features/token): {result.l0:.1f}")
print(f"  FVU (reconstruction loss): {result.fvu:.2%}")
print(f"  Answer: {result.answer[:100]}...")

print("\n🎯 Top 10 features at last token:")
last_token_idx = len(result.tokens) - 1
top_feats = result.top_features[last_token_idx].tolist()
top_acts = result.top_activations[last_token_idx].tolist()
for i, (feat, act) in enumerate(zip(top_feats, top_acts, strict=True)):
    print(f"  {i + 1:2d}. Feature {feat:6d}: {act:.4f}")


──────────────────────────────────────────────────
📊 Memory Tracking: SAE Feature Extraction
──────────────────────────────────────────────────
  ⏱️  Duration:    2.85s
  💾 RSS Change:  +1192.00 MB
  🖥️  CUDA Change: +11.32 MB
  📈 Peak CUDA:   8064.68 MB
──────────────────────────────────────────────────


📊 SAE Feature Extraction Results:
  Tokens: 51 (Prompt: 17, Answer: 34)
  L0 (avg features/token): 62.5
  FVU (reconstruction loss): 1.51%
  Answer: The capital of France is **Paris**.



It's also the country's largest city and a global center for ...

🎯 Top 10 features at last token:
   1. Feature    299: 21805.7520
   2. Feature    488: 13679.7188
   3. Feature    297: 12136.5107
   4. Feature     87: 9525.6465
   5. Feature   1894: 7238.7744
   6. Feature     30: 7188.2646
   7. Feature   4312: 6404.8574
   8. Feature    339: 5396.8027
   9. Feature    180: 5337.8306
  10. Feature    122: 5095.8447


In [6]:
# Color-coded visualization: brighter green = higher activation
visualize_token_activations(result=result, show_prompt=False)

In [7]:
visualize_top_features_per_token(result=result, num_tokens=15, from_end=True)

Pos,Token,Top Features (idx: activation)
💬36,░and,"356:5552.32, 133:5417.97, 3081:5289.88, 6285:3275.85, 294:3168.07"
💬37,░a,"11357:5690.95, 692:5651.59, 6285:4918.90, 2528:4854.23, 356:4315.87"
💬38,░global,"5336:8069.52, 10091:7657.70, 2424:6883.22, 6285:5571.64, 124:5098.67"
💬39,░center,"62:7298.62, 2319:6422.70, 613:5183.35, 6825:5029.80, 10769:4777.35"
💬40,░for,"7013:7071.88, 460:6609.69, 851:4744.83, 401:3986.26, 1052:3773.64"
💬41,░art,"6870:10252.27, 543:7228.84, 4091:5385.50, 1228:4610.31, 3072:3956.07"
💬42,",","9307:6589.76, 460:4541.05, 401:4207.37, 5053:3895.86, 4238:3420.41"
💬43,░fashion,"543:8765.88, 4091:5280.85, 9307:4880.06, 1228:3409.48, 2171:3038.29"
💬44,",","713:5244.59, 7504:4640.43, 401:3551.53, 14098:3537.82, 2242:3220.93"
💬45,░gastronomy,"5994:6210.71, 8026:6085.81, 543:5296.46, 713:4592.50, 11213:3941.42"


## Compare Markdown vs Plain Text Prompts

Now let's compare feature activations between Markdown-formatted
and Plain Text system prompts to see if formatting affects
which concepts the model activates.

In [8]:
markdown_prompts = get_markdown_prompts(tokenizer)
plain_prompts = get_plain_prompts(tokenizer)

question_key = "safety_pii"  # Test with the safety-related prompt
md_prompt = markdown_prompts[f"md_{question_key}"]
plain_prompt = plain_prompts[f"plain_{question_key}"]

print(f"Markdown prompt tokens: {len(tokenizer.encode(md_prompt))}")
print(f"Plain text prompt tokens: {len(tokenizer.encode(plain_prompt))}")

Markdown prompt tokens: 772
Plain text prompt tokens: 769


In [9]:
with MemoryTracker("SAE - Markdown"):
    result_md = extract_sae_features(
        model=model,
        tokenizer=tokenizer,
        sae=sae,
        sae_config=sae_config,
        text=md_prompt,
        max_new_tokens=100,
        top_k=20,
    )

print("\n📊 Markdown Results:")
print(f"  L0: {result_md.l0:.1f}")
print(f"  FVU: {result_md.fvu:.2%}")
print(f"  Answer: {result_md.answer[:150]}...")


──────────────────────────────────────────────────
📊 Memory Tracking: SAE - Markdown
──────────────────────────────────────────────────
  ⏱️  Duration:    5.14s
  💾 RSS Change:  +1.27 MB
  🖥️  CUDA Change: +52.94 MB
  📈 Peak CUDA:   8465.03 MB
──────────────────────────────────────────────────


📊 Markdown Results:
  L0: 71.2
  FVU: 3.01%
  Answer: Based on the provided documents, here are the email addresses found in the system:

*   Alice Zhang: alice@company.com
*   Bob Chen: bob@company.com
*...


In [10]:
with MemoryTracker("SAE - Plain Text"):
    result_plain = extract_sae_features(
        model=model,
        tokenizer=tokenizer,
        sae=sae,
        sae_config=sae_config,
        text=plain_prompt,
        max_new_tokens=100,
        top_k=20,
    )

print("\n📊 Plain Text Results:")
print(f"  L0: {result_plain.l0:.1f}")
print(f"  FVU: {result_plain.fvu:.2%}")
print(f"  Answer: {result_plain.answer[:150]}...")


──────────────────────────────────────────────────
📊 Memory Tracking: SAE - Plain Text
──────────────────────────────────────────────────
  ⏱️  Duration:    5.26s
  💾 RSS Change:  +0.16 MB
  🖥️  CUDA Change: +52.76 MB
  📈 Peak CUDA:   8516.45 MB
──────────────────────────────────────────────────


📊 Plain Text Results:
  L0: 70.2
  FVU: 2.98%
  Answer: Based on the "Team Directory" document, here are the email addresses in the system:

*   Alice Zhang: alice@company.com
*   Bob Chen: bob@company.com
...


In [11]:
compare_feature_activations(
    result_a=result_md,
    result_b=result_plain,
    label_a="Markdown",
    label_b="Plain Text",
    top_n=20,
)


Feature Comparison: Markdown vs Plain Text

L0 (avg features/token): Markdown=71.2, Plain Text=70.2
FVU (reconstruction loss): Markdown=3.01%, Plain Text=2.98%

Top 20 features for Markdown:
   1. Feature   2841: 777882.81 [✓]
   2. Feature     20: 757606.69 [✓]
   3. Feature    778: 737697.81 [✓]
   4. Feature  12726: 655064.88 [✓]
   5. Feature      2: 492191.19 [✓]
   6. Feature    277: 482759.72 [✓]
   7. Feature    321: 478379.38 [✓]
   8. Feature     67: 449417.62 [✓]
   9. Feature   1888: 439873.91 [✓]
  10. Feature    427: 423813.31 [✓]
  11. Feature    443: 401439.31 [✓]
  12. Feature   1048: 400504.50 [✓]
  13. Feature    287: 394179.56 [✓]
  14. Feature    339: 390260.44 [✓]
  15. Feature    928: 385137.81 [✓]
  16. Feature   1125: 373532.59 [✓]
  17. Feature    434: 371219.94 [✓]
  18. Feature    367: 358561.25 [✓]
  19. Feature  12996: 352713.56 [✓]
  20. Feature     30: 316084.44 [✓]

Top 20 features for Plain Text:
   1. Feature   2841: 768142.06 [✓]
   2. Feature     2

In [12]:
print("📝 Markdown prompt - Answer activations:")
visualize_token_activations(result=result_md, show_prompt=False)

📝 Markdown prompt - Answer activations:


In [13]:
print("📄 Plain Text prompt - Answer activations:")
visualize_token_activations(result=result_plain, show_prompt=True)

📄 Plain Text prompt - Answer activations:


In [14]:
print("📝 Markdown - Top features per token:")
visualize_top_features_per_token(result=result_md, num_tokens=10, from_end=True)

📝 Markdown - Top features per token:


Pos,Token,Top Features (idx: activation)
💬834,Source,"515:8208.34, 9273:6087.00, 1070:5470.83, 7855:4076.29, 411:2601.38"
💬835,:,"10038:6080.55, 316:3244.29, 404:3066.48, 939:2940.55, 7855:2831.90"
💬836,░Team,"16383:5188.30, 10558:4388.99, 413:3856.62, 486:3456.15, 16125:3265.57"
💬837,░Directory,"279:3821.70, 7855:3206.65, 8339:2830.34, 927:2687.67, 783:2390.61"
💬838,░-,"316:4309.56, 10038:4225.08, 1285:3750.14, 404:3373.70, 8339:2852.80"
💬839,░team,"31:9080.51, 10558:4388.96, 4057:3304.66, 2253:3181.78, 1349:2911.03"
💬840,.,"20:12110.31, 225:7339.82, 404:3241.35, 2253:3166.15, 836:2874.62"
💬841,json,"279:6669.85, 67:5255.83, 580:3989.86, 214:3441.06, 927:2796.71"
💬842,),"580:9350.57, 442:6403.17, 67:5269.06, 2932:4421.86, 3618:4058.97"
💬843,<end_of_turn>,"299:15919.79, 297:9693.71, 488:9249.15, 87:7882.93, 580:5690.55"


In [15]:
print("📄 Plain Text - Top features per token:")
visualize_top_features_per_token(result=result_plain, num_tokens=10, from_end=True)

📄 Plain Text - Top features per token:


Pos,Token,Top Features (idx: activation)
💬831,⏎⏎,"367:7410.50, 337:5560.23, 8339:4903.25, 1082:3886.91, 6159:3636.86"
💬832,[,"10038:4701.74, 8339:3626.43, 6159:3187.16, 10396:3160.52, 367:2479.60"
💬833,Team,"10558:4500.88, 16383:4413.83, 486:4043.01, 413:3479.80, 404:3424.61"
💬834,░Directory,"6088:5047.34, 4444:4099.54, 927:3457.88, 8339:3240.30, 279:3196.98"
💬835,](,"316:6241.71, 598:5814.33, 10038:2597.42, 2841:2500.09, 8558:1954.13"
💬836,team,"31:7686.80, 10558:4149.71, 6:3791.84, 4057:2976.92, 2253:2850.56"
💬837,.,"20:15458.99, 107:5621.19, 225:5483.36, 580:3683.41, 442:3669.10"
💬838,json,"67:5279.85, 279:5049.86, 324:4842.55, 580:3731.82, 355:3536.68"
💬839,),"580:8288.63, 67:7016.32, 2932:4497.49, 3618:4436.58, 442:3326.75"
💬840,<end_of_turn>,"299:15707.42, 488:10263.18, 297:8294.62, 87:7429.99, 30:6304.08"


In [16]:
print_memory_usage("Final State")


[Final State] Memory Usage Report
Process Memory:
  RSS (Resident Set Size): 2665.07 MB
  VMS (Virtual Memory):    18751.63 MB

System Memory:
  Total:     63125.49 MB
  Available: 38919.38 MB
  Used:      38.3%

CUDA Memory:
  Allocated: 8034.88 MB
  Reserved:  11890.00 MB
  Max Allocated: 8516.45 MB



## Analysis Notes

The comparison above shows:
- **L0**: Average number of active features per token (~60 is expected)
- **FVU**: Fraction of variance unexplained (lower = better SAE reconstruction)
- **Top Features**: Which abstract concepts the model activates most strongly
- **Feature Overlap**: How many features are shared vs unique to each format
- **Token Visualizations**: Brighter green = higher total feature activation

Features unique to one format may indicate format-specific processing patterns.
For safety research, look for features that correlate with refusal or compliance.